In [81]:
import pandas as pd
from meteostat import Hourly, Point


class WeatherPrep():
    def __init__(
        self,
        lat: float,
        lon: float,
        start,
        end,
    ):
        self._lat = lat
        self._lon = lon
        self._start = start
        self._end = end

        self._df = None

    @property
    def lat(self):
        return self._lat

    @property
    def lon(self):
        return self._lon

    @property
    def start(self):
        return self._start

    @property
    def end(self):
        return self._end

    @property
    def df(self):
        if self._df is None:
            self._df = self._prepare_df()
        return self._df

    def _prepare_df(self):
        point = Point(self.lat, self.lon)
        df = Hourly(point, self.start, self.end).fetch()
        df["lat"] = self.lat
        df["lon"] = self.lon

        if df.empty:
            raise Exception("No weather data found")

        df = df.interpolate(limit=3)
        df = df.bfill().ffill()

        df = df.reset_index()
        return df

In [82]:
import time
import requests
import pandas as pd
import numpy as np


class GeoPrep():
    def __init__(
        self,
        hotel_name: str,
        city: str = None,
        country: str = "France",
        radii_km=(1, 2, 3),
        shop_types=None,
        user_agent: str = "ml-project",
        overpass_url: str = "https://overpass-api.de/api/interpreter",
        timeout: int = 60,
        sleep_seconds: float = 1.2,
        max_retries: int = 5,
    ):
        self._hotel_name = hotel_name
        self._city = city
        self._country = country
        self._radii_km = tuple(sorted(set(radii_km)))
        self._shop_types = shop_types or (
            "convenience",
            "supermarket",
            "grocery",
            "bakery",
            "beverages",
            "alcohol",
            "confectionery",
            "ice_cream",
            "tobacco",
            "kiosk",
            "pharmacy",
            "cosmetics",
            "gift",
        )
        self._user_agent = user_agent
        self._overpass_url = overpass_url
        self._timeout = timeout
        self._sleep_seconds = sleep_seconds
        self._max_retries = max_retries

        self._location = None
        self._hotel_df = None
        self._poi_df = None
        self._hotels_df = None

    @property
    def hotel_name(self):
        return self._hotel_name

    @property
    def city(self):
        if self._city is None:
            _ = self.location
        return self._city

    @property
    def country(self):
        return self._country
    

    @property
    def shop_types(self):
        return self._shop_types

    @property
    def radii_km(self):
        return self._radii_km

    @property
    def max_radius_km(self):
        return max(self.radii_km)

    @property
    def user_agent(self):
        return self._user_agent

    @property
    def overpass_url(self):
        return self._overpass_url

    @property
    def timeout(self):
        return self._timeout

    @property
    def sleep_seconds(self):
        return self._sleep_seconds

    @property
    def max_retries(self):
        return self._max_retries

    @property
    def location(self):
        if self._location is None:
            self._location = self._geocode()
        return self._location

    @property
    def hotel_df(self):
        if self._hotel_df is None:
            loc = self.location
            self._hotel_df = pd.DataFrame([{
                "HOTEL_NAME": self.hotel_name,
                "HOTEL_CITY": self.city,
                "HOTEL_COUNTRY": self.country,
                "HOTEL_ADDRESS": loc["address"],
                "HOTEL_LAT": loc["lat"],
                "HOTEL_LON": loc["lon"],
            }])
        return self._hotel_df

    @property
    def poi_df(self):
        if self._poi_df is None:
            self._poi_df = self._prepare_poi_df()
        return self._poi_df

    @property
    def hotels_df(self):
        if self._hotels_df is None:
            self._hotels_df = self._prepare_hotels_df()
        return self._hotels_df

    def _headers(self):
        return {
            "User-Agent": self.user_agent,
            "Accept": "application/json",
        }

    def _safe_get_json(self, url, params):
        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                r = requests.get(
                    url,
                    params=params,
                    headers=self._headers(),
                    timeout=self.timeout,
                )

                status = r.status_code

                if status == 200:
                    content_type = r.headers.get("Content-Type", "")
                    if "json" in content_type.lower():
                        return r.json()
                    last_error = Exception(
                        f"Réponse non JSON. Content-Type={content_type}, body={r.text[:300]}"
                    )
                    wait = self.sleep_seconds * (2 ** (attempt - 1))
                    time.sleep(wait)
                    continue

                if status in [429, 500, 502, 503, 504]:
                    last_error = Exception(f"HTTP {status} - {r.text[:300]}")
                    wait = self.sleep_seconds * (2 ** (attempt - 1))
                    print(f"[RETRY] HTTP {status} | tentative={attempt}/{self.max_retries} | wait={wait:.1f}s")
                    time.sleep(wait)
                    continue

                raise Exception(f"HTTP {status} - {r.text[:300]}")

            except requests.exceptions.Timeout as e:
                last_error = e
                wait = self.sleep_seconds * (2 ** (attempt - 1))
                print(f"[TIMEOUT] tentative={attempt}/{self.max_retries} | wait={wait:.1f}s")
                time.sleep(wait)

            except requests.exceptions.RequestException as e:
                last_error = e
                wait = self.sleep_seconds * (2 ** (attempt - 1))
                print(f"[REQUEST ERROR] tentative={attempt}/{self.max_retries} | wait={wait:.1f}s | {e}")
                time.sleep(wait)

            except Exception as e:
                last_error = e
                if attempt == self.max_retries:
                    break
                wait = self.sleep_seconds * (2 ** (attempt - 1))
                print(f"[ERROR] tentative={attempt}/{self.max_retries} | wait={wait:.1f}s | {e}")
                time.sleep(wait)

        raise last_error

    def _extract_city_from_address(self, address_dict):
        return (
            address_dict.get("city")
            or address_dict.get("town")
            or address_dict.get("village")
            or address_dict.get("municipality")
            or address_dict.get("suburb")
            or address_dict.get("county")
        )

    def _clean_name(self, name):
        import unicodedata

        name = ''.join(
            c for c in unicodedata.normalize('NFD', name)
            if unicodedata.category(c) != 'Mn'
        )
        name = name.replace("Sacré-Cœur", "Sacre Coeur")
        name = name.replace("-", " ")
        return name

    def _geocode(self):
        url = "https://nominatim.openstreetmap.org/search"

        queries = []
        base_name = self.hotel_name
        clean_name = self._clean_name(base_name)

        queries.append(f"{base_name}, {self.country}")
        queries.append(f"{clean_name}, {self.country}")

        words = clean_name.split()
        if len(words) > 3:
            queries.append(f"{' '.join(words[:3])}, {self.country}")

        if len(words) >= 2:
            queries.append(f"{' '.join(words[-2:])}, {self.country}")

        for query in queries:
            params = {
                "q": query,
                "format": "jsonv2",
                "limit": 1,
                "addressdetails": 1,
            }

            try:
                data = self._safe_get_json(url, params)
                time.sleep(self.sleep_seconds)

                if data:
                    best = data[0]
                    address_dict = best.get("address", {})

                    if self._city is None:
                        self._city = self._extract_city_from_address(address_dict)

                    return {
                        "address": best.get("display_name"),
                        "lat": float(best["lat"]),
                        "lon": float(best["lon"]),
                    }

            except Exception:
                continue

        if self._city:
            params = {
                "q": f"{self._city}, {self.country}",
                "format": "jsonv2",
                "limit": 1,
            }

            data = self._safe_get_json(url, params)

            if data:
                best = data[0]
                return {
                    "address": best.get("display_name"),
                    "lat": float(best["lat"]),
                    "lon": float(best["lon"]),
                }

        raise Exception(f"Hotel not found after fallback: {self.hotel_name}")

    def _build_overpass_query(self, lat, lon, radius_km):
        radius_m = int(radius_km * 1000)

        query_parts = []
        for shop_type in self.shop_types:
            for osm_type in ("node", "way", "relation"):
                query_parts.append(
                    f'{osm_type}(around:{radius_m},{lat},{lon})["shop"="{shop_type}"];'
                )

        query_body = "\n".join(query_parts)

        return f"""
        [out:json][timeout:25];
        (
        {query_body}
        );
        out center tags;
        """

    def _haversine_km(self, lat1, lon1, lat2, lon2):
        lat1 = np.radians(lat1)
        lon1 = np.radians(lon1)
        lat2 = np.radians(lat2)
        lon2 = np.radians(lon2)

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
        c = 2 * np.arcsin(np.sqrt(a))
        r = 6371.0
        return c * r

    def _get_poi_max_radius(self, lat, lon):
        query = self._build_overpass_query(lat, lon, self.max_radius_km)
        data = self._safe_get_json(self.overpass_url, {"data": query})
        time.sleep(self.sleep_seconds)

        rows = []
        for el in data.get("elements", []):
            tags = el.get("tags", {})
            poi_lat = el.get("lat", el.get("center", {}).get("lat"))
            poi_lon = el.get("lon", el.get("center", {}).get("lon"))

            if poi_lat is None or poi_lon is None:
                continue

            dist_km = self._haversine_km(lat, lon, float(poi_lat), float(poi_lon))

            rows.append({
                "HOTEL_NAME": self.hotel_name,
                "HOTEL_CITY": self.city,
                "HOTEL_LAT": lat,
                "HOTEL_LON": lon,
                "OSM_TYPE": el.get("type"),
                "OSM_ID": el.get("id"),
                "NAME": tags.get("name"),
                "SHOP_TYPE": tags.get("shop"),
                "BRAND": tags.get("brand"),
                "ADDRESS_STREET": tags.get("addr:street"),
                "ADDRESS_POSTCODE": tags.get("addr:postcode"),
                "ADDRESS_CITY": tags.get("addr:city"),
                "LAT": float(poi_lat),
                "LON": float(poi_lon),
                "DIST_KM": float(dist_km),
            })

        if not rows:
            return pd.DataFrame(columns=[
                "HOTEL_NAME", "HOTEL_CITY", "HOTEL_LAT", "HOTEL_LON",
                "OSM_TYPE", "OSM_ID", "NAME", "SHOP_TYPE", "BRAND",
                "ADDRESS_STREET", "ADDRESS_POSTCODE", "ADDRESS_CITY",
                "LAT", "LON", "DIST_KM"
            ])

        df = pd.DataFrame(rows)
        df = df.drop_duplicates(subset=["OSM_TYPE", "OSM_ID"]).reset_index(drop=True)
        return df

    def _expand_radii(self, poi_base_df):
        if poi_base_df.empty:
            return pd.DataFrame(columns=[
                "HOTEL_NAME", "HOTEL_CITY", "HOTEL_LAT", "HOTEL_LON", "RAYON_KM",
                "OSM_TYPE", "OSM_ID", "NAME", "SHOP_TYPE", "BRAND",
                "ADDRESS_STREET", "ADDRESS_POSTCODE", "ADDRESS_CITY",
                "LAT", "LON", "DIST_KM"
            ])

        out = []
        for radius in self.radii_km:
            tmp = poi_base_df[poi_base_df["DIST_KM"] <= radius].copy()
            tmp["RAYON_KM"] = radius
            out.append(tmp)

        if not out:
            return pd.DataFrame()

        df = pd.concat(out, axis=0, ignore_index=True)
        df = df[[
            "HOTEL_NAME", "HOTEL_CITY", "HOTEL_LAT", "HOTEL_LON", "RAYON_KM",
            "OSM_TYPE", "OSM_ID", "NAME", "SHOP_TYPE", "BRAND",
            "ADDRESS_STREET", "ADDRESS_POSTCODE", "ADDRESS_CITY",
            "LAT", "LON", "DIST_KM"
        ]]
        return df.reset_index(drop=True)

    def _prepare_poi_df(self):
        loc = self.location
        lat = loc["lat"]
        lon = loc["lon"]

        poi_base_df = self._get_poi_max_radius(lat, lon)
        poi_df = self._expand_radii(poi_base_df)

        return poi_df.reset_index(drop=True)

    def _prepare_hotels_df(self):
        poi_df = self.poi_df.copy()

        if poi_df.empty:
            return pd.DataFrame([{
                "HOTEL_NAME": self.hotel_name,
                "HOTEL_CITY": self.city,
                "HOTEL_LAT": self.location["lat"],
                "HOTEL_LON": self.location["lon"],
            }])

        poi_df = poi_df.rename({"NAME": "SHOP_NAME"}, axis=1)
        poi_df = poi_df[
            ["HOTEL_NAME", "HOTEL_CITY", "HOTEL_LAT", "HOTEL_LON", "RAYON_KM", "SHOP_NAME", "SHOP_TYPE"]
        ].copy()

        poi_df = poi_df[poi_df["SHOP_TYPE"].notna()].copy()

        poi_df["SHOP_TYPE_RAYON_KM"] = (
            poi_df["SHOP_TYPE"].str.upper()
            + "_"
            + poi_df["RAYON_KM"].astype(str)
            + "_KM"
        )

        dummies_df = pd.get_dummies(poi_df["SHOP_TYPE_RAYON_KM"], dtype=int)

        poi_features_df = pd.concat([poi_df, dummies_df], axis=1)
        poi_features_df = poi_features_df.drop(
            ["RAYON_KM", "SHOP_NAME", "SHOP_TYPE", "SHOP_TYPE_RAYON_KM"],
            axis=1
        )

        hotels_df = poi_features_df.groupby(
            ["HOTEL_NAME", "HOTEL_CITY", "HOTEL_LAT", "HOTEL_LON"],
            as_index=False
        ).sum()

        return hotels_df

In [83]:
import pandas as pd
import numpy as np

class DataPrep():
    def __init__(self, 
        filepath : str,
        dt_col : str = "DATE",
        tm_col : str = "HEURE",
        dttm_col : str = "DATETIME",
    ):
        self._filepath = filepath
        self._dt_col = dt_col
        self._tm_col = tm_col
        self._dttm_col = dttm_col

        self._src_df = None
        self._df = None


    @property
    def filepath(self):
        return self._filepath
    

    @property
    def dt_col(self):
        return self._dt_col
    

    @property
    def tm_col(self):
        return self._tm_col
    

    @property
    def dttm_col(self):
        return self._dttm_col


    @property
    def src_df(self):
        if(self._src_df is None):
            self._src_df = pd.read_csv(self.filepath)
        return self._src_df
    


    @property
    def df(self):
        if(self._df is None):
            df = self.prepare_df(self.src_df, 
                dt_col = self.dt_col, 
                tm_col = self.tm_col, 
                dttm_col = self.dttm_col
            )
            self._df = df
        return self._df
    

    @classmethod
    def prepare_df(cls, 
        df : pd.DataFrame,
        dt_col = "DATE",
        tm_col = "HEURE",
        dttm_col = "DATETIME",
        ref_dt = "2020-01-01",
        statut_col = "STATUT",
    ):
        df = df[df[statut_col] == "DONE"]
        
        df[dttm_col] = pd.to_datetime(df[dt_col] + " " + df[tm_col], format = "mixed")

   
        df["MOIS_D_ANNEE"] = df[dttm_col].dt.month
        df["JOUR_DU_MOIS"] = df[dttm_col].dt.day
        df["JOUR_DE_SEMAINE"] = df[dttm_col].dt.dayofweek
        df["JOUR_D_ANNEE"] = df[dttm_col].dt.dayofyear
        df["SEMAINE_D_ANNEE"] = df[dttm_col].dt.isocalendar().week.astype(int)
        df["IS_WEEKEND"] = df["JOUR_DE_SEMAINE"].isin([5, 6]).astype(int)
        df["HEURE_DU_JOUR"] = df[dttm_col].dt.hour

        map_col_cycle = {
            "MOIS_D_ANNEE" : 12.0,
            "JOUR_DU_MOIS" : 31.0,
            "JOUR_DE_SEMAINE" : 7.0,
            "JOUR_D_ANNEE" : 366.0,
            "SEMAINE_D_ANNEE" : 53.0,
            "HEURE_DU_JOUR" : 24.0,
        }

        # for col, cycle in map_col_cycle.items():
        #     df[f"{col}_COS"] = np.cos(2 * np.pi * df[col] / cycle)
        #     df[f"{col}_SIN"] = np.sin(2 * np.pi * df[col] / cycle)

        
        df = df.drop([
            dt_col, tm_col, statut_col, 
            "CODE EAN", "PRIX HT", "METEO DU JOUR (MOYENNE)", "METEO DU MOIS (MOYENNE)", "TEMPERATURE",
            # "MOIS_D_ANNEE", "JOUR_DU_MOIS", "JOUR_DE_SEMAINE", "JOUR_D_ANNEE", "SEMAINE_D_ANNEE", "HEURE_DU_JOUR",
            ] 
            # + list(map_col_cycle.keys())
            , 
            axis = 1
        )

        return df

In [87]:
import pandas as pd
import numpy as np
import datetime

filename = "001.queryVentes.csv"
dp = DataPrep(filename)
dp.src_df.head(3)

,NOM BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE EAN,NOM DU PRODUIT,QUANTITE,PRIX HT,VAT,PRIX TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER ID (TICKET DE CAISSE),TEMPERATURE,METEO DU JOUR (MOYENNE),METEO DU MOIS (MOYENNE)
0,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,12:04:22.555,DONE,3.583788e+12,TONGS FEMME 100 NOIR,1,NaN,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9281,23.1,NaN,NaN
1,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,18:32:38.901,DONE,3.608439e+12,CASQUETTE ENFANT -MH100,1,NaN,20.0,12.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9282,23.1,NaN,NaN
2,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-11,12:11:15.59,DONE,3.583788e+12,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,NaN,20.0,29.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9283,24.6,NaN,NaN


In [89]:
gps = [GeoPrep(hotel_name = hotel_name) for hotel_name in dp.src_df["NOM BOUTIQUE"].unique()]
poi_df = pd.concat([gp.poi_df for gp in gps])
poi_df.to_csv("poi.csv", index = False) # 4m12s

[RETRY] HTTP 504 | tentative=1/5 | wait=1.2s
[RETRY] HTTP 504 | tentative=2/5 | wait=2.4s
[RETRY] HTTP 504 | tentative=1/5 | wait=1.2s
[RETRY] HTTP 504 | tentative=2/5 | wait=2.4s
[RETRY] HTTP 429 | tentative=1/5 | wait=1.2s
[RETRY] HTTP 429 | tentative=2/5 | wait=2.4s
[RETRY] HTTP 504 | tentative=3/5 | wait=4.8s
[RETRY] HTTP 504 | tentative=1/5 | wait=1.2s
[RETRY] HTTP 504 | tentative=2/5 | wait=2.4s
[RETRY] HTTP 504 | tentative=3/5 | wait=4.8s
[RETRY] HTTP 429 | tentative=1/5 | wait=1.2s


In [92]:
hotels_df = pd.concat([gp.hotels_df for gp in gps])
hotels_df.to_csv("hotels_df.csv", index = False) # 4m12s

In [93]:
hotels_df

,HOTEL_NAME,HOTEL_CITY,HOTEL_LAT,HOTEL_LON,ALCOHOL_1_KM,ALCOHOL_2_KM,ALCOHOL_3_KM,BAKERY_1_KM,BAKERY_2_KM,BAKERY_3_KM,...,BEVERAGES_2_KM,BEVERAGES_3_KM,CONFECTIONERY_1_KM,COSMETICS_1_KM,GIFT_1_KM,GROCERY_2_KM,GROCERY_3_KM,GROCERY_1_KM,ICE_CREAM_2_KM,ICE_CREAM_3_KM
0,Ibis budget Nice,Nice,43.667571,7.214308,1,2,2,2,16,26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,Mercure Paris Montmartre Sacré-Cœur,Paris,48.885240,2.330055,33,78,134,82,269,480,...,3.0,12.0,9.0,13.0,53.0,2.0,4.0,NaN,NaN,NaN
0,Novotel Megève Mont-Blanc,Megève,45.859850,6.619478,1,1,1,2,2,2,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
0,Novotel Paris Tour Eiffel,Paris,48.849660,2.283704,13,47,76,46,139,252,...,2.0,4.0,2.0,13.0,6.0,3.0,3.0,1.0,NaN,NaN
0,Novotel Porte d’Italie,Paris,48.819054,2.359406,4,18,46,36,108,227,...,1.0,4.0,1.0,1.0,9.0,1.0,4.0,NaN,1.0,1.0
0,Ibis budget Strasbourg Centre République,Strasbourg,48.585058,7.736591,10,19,21,40,75,121,...,1.0,1.0,8.0,18.0,12.0,3.0,3.0,NaN,NaN,NaN


In [ ]:
# hotels_df = poi_df.drop_duplicates(subset = ["HOTEL_NAME"])
# hotels_df

In [19]:
# for i in range(hotels_df.shape[0]):
#     print(hotels_df.iloc[i]["HOTEL_NAME"], hotels_df.iloc[i]["HOTEL_LON"], hotels_df.iloc[i]["HOTEL_LAT"])

In [ ]:
# wp_df = pd.concat([
#     WeatherPrep(
#         lat = hotels_df.iloc[i]["HOTEL_LAT"],
#         lon = hotels_df.iloc[i]["HOTEL_LON"],
#         start = pd.to_datetime(dp.src_df[dp.dt_col].min()),
#         end = pd.to_datetime(dp.src_df[dp.dt_col].max()) + datetime.timedelta(days=1),
#     ).df
#     for i in range(hotels_df.shape[0])
# ])# 55s

In [20]:
# wp_df.to_csv("weather.csv", index = False)

,NOM BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE EAN,NOM DU PRODUIT,QUANTITE,PRIX HT,VAT,PRIX TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER ID (TICKET DE CAISSE),TEMPERATURE,METEO DU JOUR (MOYENNE),METEO DU MOIS (MOYENNE)
0,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,12:04:22.555,DONE,3.583788e+12,TONGS FEMME 100 NOIR,1,NaN,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9281,23.1,NaN,NaN
1,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,18:32:38.901,DONE,3.608439e+12,CASQUETTE ENFANT -MH100,1,NaN,20.0,12.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9282,23.1,NaN,NaN
2,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-11,12:11:15.59,DONE,3.583788e+12,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,NaN,20.0,29.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9283,24.6,NaN,NaN


In [25]:
dp.df.head(3)

,NOM BOUTIQUE,OPERATEUR,MACHINE,NOM DU PRODUIT,QUANTITE,VAT,PRIX TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER ID (TICKET DE CAISSE),DATETIME,MOIS_D_ANNEE,JOUR_DU_MOIS,JOUR_DE_SEMAINE,JOUR_D_ANNEE,SEMAINE_D_ANNEE,IS_WEEKEND,HEURE_DU_JOUR
0,Ibis budget Nice,ADIPOS,SCANNER NICE,TONGS FEMME 100 NOIR,1,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9281,2023-08-10 12:04:22.555,8,10,3,222,32,0,12
1,Ibis budget Nice,ADIPOS,SCANNER NICE,CASQUETTE ENFANT -MH100,1,20.0,12.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9282,2023-08-10 18:32:38.901,8,10,3,222,32,0,18
2,Ibis budget Nice,ADIPOS,SCANNER NICE,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,20.0,29.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9283,2023-08-11 12:11:15.590,8,11,4,223,32,0,12


In [ ]:
poi_df = pd.read_csv("poi.csv").rename({"NAME" : "SHOP_NAME", "CITY" : "HOTEL_CITY"}, axis = 1)[["HOTEL_NAME", "HOTEL_CITY", "HOTEL_LAT", "HOTEL_LON", "RAYON_KM", "SHOP_NAME", "SHOP_TYPE"]]
poi_df["SHOP_TYPE_RAYON_KM"] = poi_df["SHOP_TYPE"].str.upper() + "_" + poi_df["RAYON_KM"].astype(str) + "_KM" 
dummies_df = pd.get_dummies(poi_df["SHOP_TYPE_RAYON_KM"], dtype = int)
poi_df = pd.concat([poi_df, dummies_df], axis = 1)
poi_df = poi_df.drop(["RAYON_KM", "SHOP_NAME", "SHOP_TYPE", "SHOP_TYPE_RAYON_KM"], axis = 1)
hotels_df = poi_df.groupby(
    [
        "HOTEL_NAME",
        "HOTEL_CITY",
        "HOTEL_LAT",
        "HOTEL_LON",
    ],
    as_index = False
).sum()
hotels_df

,HOTEL_NAME,HOTEL_CITY,HOTEL_LAT,HOTEL_LON,CONVENIENCE_1_KM,CONVENIENCE_2_KM,CONVENIENCE_3_KM,GROCERY_2_KM,GROCERY_3_KM,SUPERMARKET_1_KM,SUPERMARKET_2_KM,SUPERMARKET_3_KM,TOBACCO_1_KM,TOBACCO_2_KM,TOBACCO_3_KM
0,Ibis budget Nice,Nice,43.667571,7.214308,1,0,20,0,0,1,0,11,1,0,6
1,Ibis budget Strasbourg Centre République,Strasbourg,48.585058,7.736591,25,56,95,3,3,11,25,48,12,23,37
2,Mercure Paris Montmartre Sacré-Cœur,Paris,48.885240,2.330055,115,0,0,0,0,29,0,0,10,0,0
3,Novotel Megève Mont-Blanc,Megève,45.859850,6.619478,0,5,5,0,0,0,1,1,0,0,0
4,Novotel Paris Tour Eiffel,Paris,48.849660,2.283704,0,177,0,3,0,0,68,0,0,26,0


,HOTEL_NAME,HOTEL_CITY,HOTEL_LAT,HOTEL_LON,CONVENIENCE_1_KM,CONVENIENCE_2_KM,CONVENIENCE_3_KM,GROCERY_2_KM,GROCERY_3_KM,SUPERMARKET_1_KM,SUPERMARKET_2_KM,SUPERMARKET_3_KM,TOBACCO_1_KM,TOBACCO_2_KM,TOBACCO_3_KM
0,Ibis budget Nice,Nice,43.667571,7.214308,1,0,20,0,0,1,0,11,1,0,6
1,Ibis budget Strasbourg Centre République,Strasbourg,48.585058,7.736591,25,56,95,3,3,11,25,48,12,23,37
2,Mercure Paris Montmartre Sacré-Cœur,Paris,48.885240,2.330055,115,0,0,0,0,29,0,0,10,0,0
3,Novotel Megève Mont-Blanc,Megève,45.859850,6.619478,0,5,5,0,0,0,1,1,0,0,0
4,Novotel Paris Tour Eiffel,Paris,48.849660,2.283704,0,177,0,3,0,0,68,0,0,26,0


In [60]:
out.shape

(818, 11)

In [28]:
weather_df = pd.read_csv("weather.csv")
weather_df.head(3)

,time,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco,lat,lon
0,2023-08-10 00:00:00,20.8,16.0,74.0,0.0,0.0,330.0,11.2,17.0,1017.0,0.0,2.0,43.667571,7.214308
1,2023-08-10 01:00:00,20.2,14.5,70.0,0.0,0.0,330.0,13.0,9.3,1017.1,0.0,2.0,43.667571,7.214308
2,2023-08-10 02:00:00,20.0,13.9,68.0,0.0,0.0,330.0,16.6,11.1,1017.3,0.0,2.0,43.667571,7.214308


count      818
unique       2
top       node
freq       777
Name: OSM_TYPE, dtype: object